# Lab 3 - Model Comparison (F1)

**Framing:** Regression to predict driver race points.

**Business question:** How many constructor points should we expect from each driver this Sunday based only on pre-race information?

**Primary metric:** MAE (lower is better).

**Validation:** Temporal holdout (train: past seasons, test: future season).

In [1]:
import sys
import random
import warnings

import numpy as np
import pandas as pd
import requests

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy : {np.__version__}")
print(f"Seed  : {RANDOM_SEED}")

Python: 3.12.3
NumPy : 2.4.3
Seed  : 414


In [2]:
BASE_URL = "https://api.jolpi.ca/ergast/f1"
SEASONS = [2022, 2023, 2024]
TARGET = "points"


def paginate(url_template: str, table_key: str, row_key: str) -> list:
    rows = []
    offset = 0
    limit = 100
    while True:
        url = f"{url_template}?limit={limit}&offset={offset}"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()["MRData"]
        rows.extend(data[table_key][row_key])
        total = int(data["total"])
        offset += limit
        if offset >= total:
            break
    return rows


def time_to_sec(t: str | None) -> float | None:
    if not t:
        return None
    try:
        m, s = t.split(":")
        return int(m) * 60 + float(s)
    except Exception:
        return None


def get_season_results(year: int) -> pd.DataFrame:
    rows = []
    races = paginate(f"{BASE_URL}/{year}/results.json", "RaceTable", "Races")
    for race in races:
        for r in race["Results"]:
            rows.append({
                "season": int(race["season"]),
                "round": int(race["round"]),
                "race_name": race["raceName"],
                "date": race["date"],
                "circuit": race["Circuit"]["circuitId"],
                "driver": r["Driver"]["driverId"],
                "driver_name": f"{r['Driver']['givenName']} {r['Driver']['familyName']}",
                "constructor": r["Constructor"]["constructorId"],
                "grid": int(r["grid"]),
                "position": int(r["position"]) if r["position"].isdigit() else np.nan,
                "points": float(r["points"]),
            })
    return pd.DataFrame(rows)


def get_qualifying(year: int) -> pd.DataFrame:
    rows = []
    races = paginate(f"{BASE_URL}/{year}/qualifying.json", "RaceTable", "Races")
    for race in races:
        for r in race["QualifyingResults"]:
            rows.append({
                "season": int(race["season"]),
                "round": int(race["round"]),
                "driver": r["Driver"]["driverId"],
                "quali_pos": int(r["position"]),
                "q3_time_sec": time_to_sec(r.get("Q3")),
            })
    return pd.DataFrame(rows)


all_results = []
all_qualifying = []
print("Loading data...")
for year in SEASONS:
    yr_results = get_season_results(year)
    yr_qualifying = get_qualifying(year)
    print(f"{year}: results={len(yr_results)} | qualifying={len(yr_qualifying)}")
    all_results.append(yr_results)
    all_qualifying.append(yr_qualifying)

df_results = pd.concat(all_results, ignore_index=True)
df_qualifying = pd.concat(all_qualifying, ignore_index=True)

df = df_results.merge(
    df_qualifying[["season", "round", "driver", "quali_pos", "q3_time_sec"]],
    on=["season", "round", "driver"],
    how="left",
)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["driver", "date", "season", "round"]).reset_index(drop=True)

# Strict temporal feature engineering: only prior races are used.
df["driver_prev_points"] = df.groupby("driver")["points"].shift(1)
df["driver_prev_grid"] = df.groupby("driver")["grid"].shift(1)
df["driver_points_avg_3"] = (
    df.groupby("driver")["points"]
      .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)
df["driver_top10_rate_3"] = (
    df.assign(top10=(df["position"].fillna(99) <= 10).astype(int))
      .groupby("driver")["top10"]
      .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

df_model = df[[
    "season", "round", "race_name", "date", "driver", "driver_name", "constructor", "circuit",
    "grid", "quali_pos", "q3_time_sec",
    "driver_prev_points", "driver_prev_grid", "driver_points_avg_3", "driver_top10_rate_3",
    TARGET,
]].copy()

print(f"\nRows in modeling dataset: {len(df_model)}")
print(df_model.head(3))
print("\nMissing values in features:")
print(df_model.drop(columns=["season", "round", "race_name", "date", "driver", "driver_name", TARGET]).isna().sum().sort_values(ascending=False))

Loading data...
2022: results=440 | qualifying=440
2023: results=440 | qualifying=440
2024: results=479 | qualifying=479

Rows in modeling dataset: 1359
   season  round                 race_name       date driver      driver_name  \
0    2022      1        Bahrain Grand Prix 2022-03-20  albon  Alexander Albon   
1    2022      2  Saudi Arabian Grand Prix 2022-03-27  albon  Alexander Albon   
2    2022      3     Australian Grand Prix 2022-04-10  albon  Alexander Albon   

  constructor      circuit  grid  quali_pos  q3_time_sec  driver_prev_points  \
0    williams      bahrain    14         14          NaN                 NaN   
1    williams       jeddah    16         17          NaN                 0.0   
2    williams  albert_park    20         16          NaN                 0.0   

   driver_prev_grid  driver_points_avg_3  driver_top10_rate_3  points  
0               NaN                  NaN                  NaN     0.0  
1              14.0                  0.0                 

## Temporal split and comparison setup

- Train seasons: 2022 and 2023
- Test season: 2024
- Primary metric for every row: MAE
- Two baselines are included:
  1. Global mean points baseline
  2. Domain heuristic baseline from train average points by grid position

In [3]:
TRAIN_SEASONS = [2022, 2023]
TEST_SEASON = 2024

train_df = df_model[df_model["season"].isin(TRAIN_SEASONS)].copy()
test_df = df_model[df_model["season"] == TEST_SEASON].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Temporal split failed: one of the sets is empty.")

feature_cols_num = [
    "grid", "quali_pos", "q3_time_sec",
    "driver_prev_points", "driver_prev_grid", "driver_points_avg_3", "driver_top10_rate_3",
]
feature_cols_cat = ["constructor", "circuit"]
all_feature_cols = feature_cols_num + feature_cols_cat

X_train = train_df[all_feature_cols]
y_train = train_df[TARGET]
X_test = test_df[all_feature_cols]
y_test = test_df[TARGET]


def evaluate_regressor(name: str, model) -> dict:
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    return {
        "Model": name,
        "Validation": "Train 2022-2023 -> Test 2024",
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Gap (Test-Train)": test_mae - train_mae,
    }


def evaluate_baseline_constant() -> dict:
    pred_value = y_train.mean()
    pred_train = np.full(shape=len(y_train), fill_value=pred_value)
    pred_test = np.full(shape=len(y_test), fill_value=pred_value)
    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    return {
        "Model": "Baseline 1 - Global mean points",
        "Validation": "Train 2022-2023 -> Test 2024",
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Gap (Test-Train)": test_mae - train_mae,
    }


def evaluate_baseline_grid_heuristic() -> dict:
    # Domain heuristic: expected points by starting grid position, learned only from train seasons.
    grid_means = train_df.groupby("grid")[TARGET].mean()
    fallback = y_train.mean()

    pred_train = train_df["grid"].map(grid_means).fillna(fallback)
    pred_test = test_df["grid"].map(grid_means).fillna(fallback)

    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    return {
        "Model": "Baseline 2 - Grid heuristic",
        "Validation": "Train 2022-2023 -> Test 2024",
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Gap (Test-Train)": test_mae - train_mae,
    }


preprocess_linear = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), feature_cols_num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat),
    ]
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_cols_num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat),
    ]
)

linreg = Pipeline([
    ("prep", preprocess_linear),
    ("model", LinearRegression()),
])

ridge = Pipeline([
    ("prep", preprocess_linear),
    ("model", Ridge(alpha=1.0, random_state=RANDOM_SEED)),
])

rf = Pipeline([
    ("prep", preprocess_tree),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

rows = []
rows.append(evaluate_baseline_constant())
rows.append(evaluate_baseline_grid_heuristic())
rows.append(evaluate_regressor("Linear Regression", linreg))
rows.append(evaluate_regressor("Ridge (alpha=1.0)", ridge))
rows.append(evaluate_regressor("Random Forest (300 trees)", rf))

comparison_df = pd.DataFrame(rows).sort_values("Test MAE", ascending=True).reset_index(drop=True)
comparison_df.insert(0, "Rank", np.arange(1, len(comparison_df) + 1))

for col in ["Train MAE", "Test MAE", "Gap (Test-Train)"]:
    comparison_df[col] = comparison_df[col].round(3)

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
comparison_df

Train rows: 880 | Test rows: 479


,Rank,Model,Validation,Train MAE,Test MAE,Gap (Test-Train)
0,1,Random Forest (300 trees),Train 2022-2023 -> Test 2024,1.789,2.760,0.971
1,2,Baseline 2 - Grid heuristic,Train 2022-2023 -> Test 2024,3.471,3.092,-0.379
2,3,Ridge (alpha=1.0),Train 2022-2023 -> Test 2024,3.119,3.377,0.258
3,4,Linear Regression,Train 2022-2023 -> Test 2024,3.117,3.399,0.282
4,5,Baseline 1 - Global mean points,Train 2022-2023 -> Test 2024,5.917,5.912,-0.006


In [4]:
def reason_for_model(model_name: str) -> str:
    if model_name == "Baseline 1 - Global mean points":
        return "Uses one constant for every driver/race. It ignores grid and team context, so it should be stable but usually inaccurate."
    if model_name == "Baseline 2 - Grid heuristic":
        return "Domain baseline: starting grid has strong relation with final points, so this captures a useful prior with zero model complexity."
    if model_name == "Linear Regression":
        return "Linear effects are easy to estimate and robust, but they miss nonlinear jumps (for example, the sharp reward difference between front rows and midfield)."
    if model_name == "Ridge (alpha=1.0)":
        return "Adds regularization to linear regression, typically reducing variance. It helps when one-hot features are sparse and noisy."
    if model_name == "Random Forest (300 trees)":
        return "Captures nonlinear interactions among grid, recent form, and circuit/constructor context, but can overfit if train error becomes much lower than test."
    return "Model-specific behavior not documented."

comparison_with_why = comparison_df.copy()
comparison_with_why["WHY this result"] = comparison_with_why["Model"].map(reason_for_model)
comparison_with_why

,Rank,Model,Validation,Train MAE,Test MAE,Gap (Test-Train),WHY this result
0,1,Random Forest (300 trees),Train 2022-2023 -> Test 2024,1.789,2.760,0.971,"Captures nonlinear interactions among grid, re..."
1,2,Baseline 2 - Grid heuristic,Train 2022-2023 -> Test 2024,3.471,3.092,-0.379,Domain baseline: starting grid has strong rela...
2,3,Ridge (alpha=1.0),Train 2022-2023 -> Test 2024,3.119,3.377,0.258,"Adds regularization to linear regression, typi..."
3,4,Linear Regression,Train 2022-2023 -> Test 2024,3.117,3.399,0.282,Linear effects are easy to estimate and robust...
4,5,Baseline 1 - Global mean points,Train 2022-2023 -> Test 2024,5.917,5.912,-0.006,Uses one constant for every driver/race. It ig...


## Interpretation

- The best model should be selected by **lowest Test MAE**, not by train score.
- The **Train-Test gap** diagnoses overfitting risk.
- If a baseline wins, that is still a valid and useful result: it means current features/models are not adding enough predictive value yet.